### Full training pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip drive/MyDrive/archive.zip

In [ ]:
!pip install monai

#### 1. Data loading

In [4]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image

In [5]:
def process_csv(file_path):
    df = pd.read_csv(file_path)

    images_dict = {}

    for i in range(len(df)):
        id_full = df.iloc[i]['ID']
        label_value = df.iloc[i]['Label']

        # split the image's ID from the hemorrhage's type
        # ex: "ID_xxx_epidural" -> "ID_xxx" and "epidural"
        parts = id_full.split('_')
        hemorrhage_type = parts[-1]
        image_id = '_'.join(parts[:-1])

        # if the image is not in the dictionary, add it
        if image_id not in images_dict:
            images_dict[image_id] = {}

        # add label for the corresponding hemorrhage's type
        images_dict[image_id][hemorrhage_type] = label_value

    return images_dict

In [6]:
class RSNAHemorrhageDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.images_dict = process_csv(csv_file)
        self.image_ids = list(self.images_dict.keys())
        self.label_columns = ['epidural', 'intraparenchymal', 'intraventricular',
                             'subarachnoid', 'subdural', 'any']

        print(f"Dataset initialized with {len(self)} images from {img_dir}")
        print(f"Categories: {self.label_columns}")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # get the image's ID
        img_id = self.image_ids[idx]

        # image's path
        img_name = img_id + '_frame0.png'
        img_path = os.path.join(self.img_dir, img_name)

        # data for MONAI is in the dictionary form
        data = {'image': img_path}

        # if there are any transformations, apply them
        if self.transform:
            data = self.transform(data)

        image = data['image']

        # extract the labels
        labels = []
        for label_col in self.label_columns:
            label_value = self.images_dict[img_id].get(label_col, 0)
            labels.append(label_value)

        # convert labels to tensor
        label = torch.tensor(labels, dtype=torch.float32)

        return image, label

    def get_image_id(self, idx):
        return self.image_ids[idx]

#### 2. Preprocessing data pipeline

In [ ]:
import numpy as np
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityd,
    MedianSmoothd,
    AdjustContrastd,
    Resized,
    RandFlipd,
    RandRotated,
    RandZoomd,
    ToTensord,
)

In [8]:
class CTPreprocessingPipeline:
    def __init__(self, radius, spatial_size, gamma):
        self.radius = radius
        self.spatial_size = spatial_size
        self.gamma = gamma

        self.train_pipeline = Compose([
          LoadImaged(keys='image'),
          EnsureChannelFirstd(keys='image'),
          ScaleIntensityd(keys='image'),                       # scale pixels' intensity in [0,1]
          MedianSmoothd(keys='image', radius=self.radius),     # reduce image's noise
          AdjustContrastd(keys='image', gamma=self.gamma),     # highlight contrast
          Resized(keys='image', spatial_size=self.spatial_size, mode='bilinear'),

          # augmentation
          RandFlipd(keys='image', prob=0.5, spatial_axis=0),
          RandRotated(keys='image', range_x=np.pi/12, prob=0.5, mode='bilinear'),
          RandZoomd(keys='image', min_zoom=0.9, max_zoom=1.1, prob=0.5, mode='bilinear'),
          ToTensord(keys='image')
      ])

        self.val_pipeline = Compose([
            LoadImaged(keys='image'),
            EnsureChannelFirstd(keys='image'),
            ScaleIntensityd(keys='image'),
            MedianSmoothd(keys='image', radius=self.radius),
            AdjustContrastd(keys='image', gamma=self.gamma),
            Resized(keys='image', spatial_size=self.spatial_size, mode='bilinear'),
            ToTensord(keys='image')
        ])

        self.test_pipeline = Compose([
            LoadImaged(keys='image'),
            EnsureChannelFirstd(keys='image'),
            ScaleIntensityd(keys='image'),
            Resized(keys='image', spatial_size=self.spatial_size, mode='bilinear'),
            ToTensord(keys='image')
        ])

    def get_train_pipeline(self):
        return self.train_pipeline

    def get_val_pipeline(self):
        return self.val_pipeline

    def get_test_pipeline(self):
        return self.test_pipeline

In [ ]:
preprocessing_pipeline = CTPreprocessingPipeline(
    radius=1,
    spatial_size=(128, 128),
    gamma = 1.5
)

train_transform = preprocessing_pipeline.get_train_pipeline()
train_dataset = RSNAHemorrhageDataset(
    csv_file='subdataset_train.csv',
    img_dir='rsna-intracranial-hemorrhage-detection-png/train_images',
    transform=train_transform
)

test_transform = preprocessing_pipeline.get_test_pipeline()
test_dataset = RSNAHemorrhageDataset(
    csv_file='subdataset_test.csv',
    img_dir='rsna-intracranial-hemorrhage-detection-png/test_images',
    transform=test_transform
)

In [10]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

In [11]:
def split_dataset(dataset, test_size, random_state):
  indices = list(range(len(train_dataset)))

  train_indices, val_indices = train_test_split(
      indices,
      test_size=test_size,
      random_state=random_state,
  )

  val_transform = preprocessing_pipeline.get_val_pipeline()

  val_dataset = RSNAHemorrhageDataset(
      csv_file='subdataset_train.csv',
      img_dir='rsna-intracranial-hemorrhage-detection-png/train_images',
      transform=val_transform
  )

  print(f"Validation dataset: {len(val_dataset)} images")

  train_subset = Subset(train_dataset, train_indices)
  val_subset = Subset(val_dataset, val_indices)

  return train_subset, val_subset

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 4

train_subset, val_subset = split_dataset(train_dataset, test_size=0.2, random_state=42)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

#### 3. EfficientNet-B7

In [13]:
import torchvision.models as models
import torch.nn as nn

In [14]:
class IntracranialHemorrhageEfficientNet(nn.Module):
    NUM_CLASSES = 6

    def __init__(self, num_classes=NUM_CLASSES, pretrained=True):
        super(IntracranialHemorrhageEfficientNet, self).__init__()
        self.efficientnet = models.efficientnet_b7(pretrained=pretrained)

        original_input = self.efficientnet.features[0][0]
        self.efficientnet.features[0][0] = nn.Conv2d(
            in_channels=1,
            out_channels=original_input.out_channels,
            kernel_size=original_input.kernel_size,
            stride=original_input.stride,
            padding=original_input.padding,
            bias=False
        )

        num_features = self.efficientnet.classifier[1].in_features

        self.efficientnet.classifier = nn.Linear(num_features, num_classes)

    def forward(self, x):
      x = self.efficientnet(x)
      return x

#### 4. Loss function & optimizer

In [15]:
import torch
import torchvision.models as models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = IntracranialHemorrhageEfficientNet(num_classes=6, pretrained=True)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()  # loss function

LEARNING_RATE = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE) # optimizer

#### 5. Training loop

In [17]:
def calculate_accuracy(outputs, labels):
    probabilities = torch.sigmoid(outputs)
    predictions = (probabilities > 0.5).float()
    predicted_rows = (predictions == labels).all(dim=1).sum().item()

    return predicted_rows / labels.size(0)

In [18]:
def validate_model(model, val_loader, criterion, device):
    model.eval()
    total_val_loss = 0
    total_val_accuracy = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_val_loss += loss.item()
            total_val_accuracy += calculate_accuracy(outputs, labels)

    avg_val_loss = total_val_loss / len(val_loader)
    avg_val_accuracy = total_val_accuracy / len(val_loader)

    return avg_val_loss, avg_val_accuracy

In [19]:
best_val_accuracy = 0.0
BEST_MODEL_PATH= 'best_model.pth'

In [20]:
def save_best_model(model, optimizer, epoch, avg_train_loss, avg_val_loss,
                   avg_train_accuracy, avg_val_accuracy):
    global best_val_accuracy

    if avg_val_accuracy > best_val_accuracy:
        best_val_accuracy = avg_val_accuracy
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'train_accuracy': avg_train_accuracy,
            'val_accuracy': avg_val_accuracy,
            'best_val_accuracy': best_val_accuracy
        }, BEST_MODEL_PATH)


In [21]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                num_epochs, device):
    train_loss_history = []
    val_loss_history = []
    train_accuracy_history = []
    val_accuracy_history = []

    for epoch in range(1, num_epochs + 1):
        # model's training
        model.train()
        total_train_loss = 0
        total_train_accuracy = 0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            total_train_accuracy += calculate_accuracy(outputs, labels)

            # show the current progress
            if (batch_idx + 1) % 10 == 0:
                print(f"Epoch [{epoch}/{num_epochs}], "
                      f"Batch [{batch_idx + 1}/{len(train_loader)}], "
                      f"Loss: {loss.item():.4f}, "
                      f"Acc: {calculate_accuracy(outputs, labels):.4f}")


        avg_train_loss = total_train_loss / len(train_loader)
        avg_train_accuracy = total_train_accuracy / len(train_loader)

        # model's validation
        avg_val_loss, avg_val_accuracy = validate_model(
            model, val_loader, criterion, device
        )

        # keep the results
        train_loss_history.append(avg_train_loss)
        train_accuracy_history.append(avg_train_accuracy)
        val_loss_history.append(avg_val_loss)
        val_accuracy_history.append(avg_val_accuracy)

        # epoch's summary
        print(f"\nEpoch [{epoch}/{num_epochs}] Summary:")
        print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_accuracy:.4f}")
        print(f"Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_accuracy:.4f}")

        # save best model
        save_best_model(model, optimizer, epoch, avg_train_loss, avg_val_loss,
                       avg_train_accuracy, avg_val_accuracy)

    print("Training completed!")
    print(f"Best validation accuracy: {best_val_accuracy:.4f}")

    # for plotting
    return {
        'train_loss': train_loss_history,
        'val_loss': val_loss_history,
        'train_accuracy': train_accuracy_history,
        'val_accuracy': val_accuracy_history
    }

##### Plots

In [22]:
import matplotlib.pyplot as plt

In [23]:
def plot_loss_evolution(history):
    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(epochs, history['train_loss'], 'b-o', label='Training Loss', linewidth=2)
    plt.plot(epochs, history['val_loss'], 'r-s', label='Validation Loss', linewidth=2)

    plt.title('Loss Evolution per Epoch', fontsize=16, fontweight='bold')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.show()

In [24]:
def plot_accuracy_evolution(history):
    epochs = range(1, len(history['train_accuracy']) + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(epochs, history['train_accuracy'], 'b-o', label='Training Accuracy', linewidth=2)
    plt.plot(epochs, history['val_accuracy'], 'r-s', label='Validation Accuracy', linewidth=2)

    plt.title('Accuracy Evolution per Epoch', fontsize=16, fontweight='bold')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.show()

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=15,
    device=device
)

plot_loss_evolution(history)
plot_accuracy_evolution(history)

#### 6. Evaluation

In [26]:
def test_model(model, test_loader, criterion, device):
    model.eval()
    total_test_loss = 0
    total_test_accuracy = 0

    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(test_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_test_loss += loss.item()
            total_test_accuracy += calculate_accuracy(outputs, labels)

    avg_test_loss = total_test_loss / len(test_loader)
    avg_test_accuracy = total_test_accuracy / len(test_loader)

    print(f"Test Results:")
    print(f"Test Loss: {avg_test_loss:.4f}")
    print(f"Test Accuracy: {avg_test_accuracy:.4f}")

    return avg_test_loss, avg_test_accuracy

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH)
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_accuracy = test_model(
    model=model,
    test_loader=test_loader,
    criterion=criterion,
    device=device
)

In [28]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [29]:
def evaluate_model(model, test_loader, device, threshold=0.5):
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            probabilities = torch.sigmoid(outputs)
            predictions = (probabilities > threshold).float()

            all_predictions.append(predictions.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_probabilities.append(probabilities.cpu().numpy())

    all_predictions = np.concatenate(all_predictions, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_probabilities = np.concatenate(all_probabilities, axis=0)

    return all_predictions, all_labels, all_probabilities

In [ ]:
predictions, labels, probabilities = evaluate_model(model, test_loader, device)

precision, recall, f1, _ = precision_recall_fscore_support(
    labels, predictions, average='weighted'
)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

#### Confusion matrices

In [31]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

In [32]:
def plot_overall_confusion_matrix(predictions, labels, figsize=(6, 5), cmap='Blues'):
    flat_predictions = predictions.flatten()
    flat_labels = labels.flatten()

    cm = confusion_matrix(flat_labels, flat_predictions)

    plt.figure(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt='g', cmap=cmap,
                xticklabels=['Predicted Negative', 'Predicted Positive'],
                yticklabels=['Actual Negative', 'Actual Positive'])
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.title('Overall Confusion Matrix (Flattened)')
    plt.tight_layout()
    plt.show()

    return cm

In [ ]:
overall_cm = plot_overall_confusion_matrix(predictions, labels, cmap='Blues')

In [36]:
def plot_per_label_confusion_matrices(predictions, labels, class_names,
                                      figsize_per_subplot=(6, 5), cmap='Blues'):
    results = {}

    # calculate and display accuracies
    accuracies = {}
    for i, class_name in enumerate(class_names):
        y_true = labels[:, i]
        y_pred = predictions[:, i]
        acc = accuracy_score(y_true, y_pred)
        accuracies[class_name] = acc
        print(f"{class_name.capitalize()}: {acc:.4f}")

    # grid dimensions
    n_labels = len(class_names)
    n_cols = min(3, n_labels)                   # max 3 columns
    n_rows = (n_labels + n_cols - 1) // n_cols  # ceiling division

    # create figure with subplots
    fig, axes = plt.subplots(n_rows, n_cols,
                            figsize=(figsize_per_subplot[0] * n_cols,
                                    figsize_per_subplot[1] * n_rows))

    # adjust spacing between subplots
    plt.subplots_adjust(hspace=0.4, wspace=0.3)

    # flatten axes array for easier iteration
    if n_labels == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if n_labels > n_cols else axes

    # generate confusion matrices
    for i, class_name in enumerate(class_names):
        y_true = labels[:, i]
        y_pred = predictions[:, i]

        # calculate confusion matrix
        cm = confusion_matrix(y_true, y_pred)

        # store results
        results[class_name] = {
            'accuracy': accuracies[class_name],
            'confusion_matrix': cm
        }

        # plot confusion matrix on subplot
        sns.heatmap(cm, annot=True, fmt='g', cmap=cmap,
                    xticklabels=['Predicted Negative', 'Predicted Positive'],
                    yticklabels=['Actual Negative', 'Actual Positive'],
                    ax=axes[i], cbar_kws={'shrink': 0.8})

        axes[i].set_xlabel('Predicted label')
        axes[i].set_ylabel('True label')
        axes[i].set_title(f'{class_name.capitalize()}\nAccuracy: {accuracies[class_name]:.4f}')

    # hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

    return results

In [ ]:
class_names = ['epidural', 'intraparenchymal', 'intraventricular',
               'subarachnoid', 'subdural']

# per-label confusion matrices
per_label_results = plot_per_label_confusion_matrices(
    predictions,
    labels,
    class_names
)